In [133]:
from glob import glob
import os
import pandas as pd
import numpy as np

# Configuração dos caminhos
caminho_videos = 'Audios Tunad/videos'
caminho_attentive = os.path.join(caminho_videos, 'attentiveai')

# --- CORREÇÃO 1: Função para limpar o nome do arquivo e usar como Chave ---
def carregar_dados(pattern, suffix_to_remove):
    arquivos = sorted(glob(pattern))
    dados = {}
    for caminho in arquivos:
        # Pega apenas o nome do arquivo (sem pastas)
        nome_base = os.path.basename(caminho)
        # Remove o sufixo (.npy, .foco.npy, etc) para sobrar apenas "video.mp4"
        chave = nome_base.replace(suffix_to_remove, '')
        dados[chave] = np.load(caminho)
    return dados

# Carregando os dicionários com as chaves limpas
# Ex: Chave será '00a1b...mp4' ao invés do caminho completo
print("Carregando arquivos .npy...")
curvas = carregar_dados(os.path.join(caminho_videos, '*.npy'), '.npy')
focos = carregar_dados(os.path.join(caminho_attentive, '*.foco.npy'), '.foco.npy')
demandas = carregar_dados(os.path.join(caminho_attentive, '*.demandacognitiva.npy'), '.demandacognitiva.npy')

# Carregamos a planilha

df = pd.read_excel('Audios Tunad/Audios Tunad ATUALIZADO.xlsx')

# Renomeando colunas (garantindo que existem colunas suficientes)
colunas_desejadas = ['Audio', 'TotalInsertions', 'TotalLift', 'AvgLift', 'AvgCallToAction', 
                        'AvgEffectiveMinutes', 'AvgUpliftProportional', 'Nome do Comercial', 
                        'AdBrandName', 'Secundagem', 'Tags', 'Link do vídeo']

if len(df.columns) == len(colunas_desejadas):
    df.columns = colunas_desejadas
else:
    print(f"Aviso: O número de colunas no Excel ({len(df.columns)}) difere do esperado ({len(colunas_desejadas)}).")

# --- CORREÇÃO 2: Limpeza do Link (Vectorized) ---
# Transforma 'https://.../video.mp4' em 'video.mp4'
df['Link do vídeo'] = df['Link do vídeo'].astype(str).str.split('/').str[-1]

# Função auxiliar para calcular média segura (evita erro se a chave não existir)
def get_mean(dic_dados, chave):
    if chave in dic_dados:
        return dic_dados[chave].mean()
    return np.nan # Retorna vazio se não achar o arquivo

def get_mean_sec(dic_dados, chave, sec):
    if chave in dic_dados:
        tamanho = len(dic_dados[chave])
        return dic_dados[chave][:min([2*sec + 1, tamanho -1])].mean()
    return 0.1 # Retorna vazio se não achar o arquivo

print("Mapeando dados...")
# --- CORREÇÃO 3: Mapping seguro ---
df['engneural'] = df['Link do vídeo'].apply(lambda x: get_mean(curvas, x))
df['foco'] = df['Link do vídeo'].apply(lambda x: get_mean(focos, x))
df['demanda_cognitiva'] = df['Link do vídeo'].apply(lambda x: get_mean(demandas, x))


df['engneural_5sec'] = df['Link do vídeo'].apply(lambda x: get_mean_sec(curvas, x, 5))
df['foco_5sec'] = df['Link do vídeo'].apply(lambda x: get_mean_sec(focos, x, 5))
df['demanda_cognitiva_5sec'] = df['Link do vídeo'].apply(lambda x: get_mean_sec(demandas, x, 5))

# Acrescentar o log do lift
# Usando np.log1p que é equivalente a log(x + 1) e mais preciso
df['LOG(TotalLift+1)'] = np.log1p(df['TotalLift'])
df['LOG(AvgLift+1)'] = np.log1p(df['AvgLift'])
df['LOG(AvgUpliftProportional+1)'] = np.log1p(df['AvgUpliftProportional'])
df['final'] = np.log1p(df['AvgLift'])

df.dropna(inplace=True)
df = df[df['TotalLift'] > 0]
print(f"Processamento concluído. Total de linhas: {len(df)}")
print(df[['Link do vídeo', 'engneural', 'foco', 'demanda_cognitiva', 'engneural_5sec', 'foco_5sec', 'demanda_cognitiva_5sec']].head())
print(df.head())


Carregando arquivos .npy...
Mapeando dados...
Processamento concluído. Total de linhas: 307
                          Link do vídeo  engneural       foco  \
0  6b53304be2f4bb0a8c084fca52927e7b.mp4  76.568127  43.783424   
1  0402a20bae8bece3ffda24c9d55e46be.mp4  45.717038  58.137487   
2  40616f6c73afca358421b06677183614.mp4  52.678702  59.046305   
3  18f2f97fb58633b48c511cd91bbcb523.mp4  46.096494  49.695417   
4  8f553b31caaeffa8d6ff21ad9daa4fd4.mp4  65.829393  65.107659   

   demanda_cognitiva  engneural_5sec  foco_5sec  demanda_cognitiva_5sec  
0          49.063420       74.632745  49.593627               48.171965  
1          36.879001       49.223679  42.501242               33.642100  
2          49.832650       76.872875  38.853814               44.366464  
3          41.211382       60.660031  58.981642               44.133248  
4          65.060452       77.160422  58.329648               65.367951  
    Audio  TotalInsertions  TotalLift   AvgLift  AvgCallToAction  \
0  83

In [ ]:
import pandas as pd
from collections import Counter

# --- Simulação do seu DataFrame (pode ignorar esta parte) ---
# df = pd.DataFrame(...) 

# 1. PRÉ-PROCESSAMENTO
# Garantir que é string, preencher nulos e fazer o split por vírgula
# O .strip() remove espaços em branco extras no início/fim de cada tag
df['lista_tags'] = df['Tags'].fillna('').astype(str).apply(lambda x: [item.strip() for item in x.split(',') if item.strip() != ''])

# 2. IDENTIFICAR AS TOP 10 TAGS
# Juntamos todas as listas em uma só e contamos a frequência
todas_tags = [tag for lista in df['lista_tags'] for tag in lista]
top_10_tags = [tag for tag, count in Counter(todas_tags).most_common(10)]

print(f"As 10 tags mais frequentes são: {top_10_tags}")

# 3. APLICAR ONE HOT ENCODING (MANUAL)
# Para cada uma das top 10 tags, criamos uma coluna nova
# Se a tag estiver na lista daquela linha, recebe 1, senão 0
for tag in top_10_tags:
    # Criamos um nome de coluna seguro (opcional: prefixo 'tag_')
    col_name = f"Tag_{tag}"
    df[col_name] = df['lista_tags'].apply(lambda x: 1 if tag in x else 0)

# (Opcional) Remover a coluna auxiliar de lista se não precisar mais
# df = df.drop(columns=['lista_tags'])

# Visualizar o resultado
print(df.columns)

As 10 tags mais frequentes são: ['Emotions Emotions:informal', 'Tone:Informal', 'Emotions Sensations:Informal', 'Characters Gender:Masculino', 'Detalhe de Produto / Serviço:masculino', 'Emotions Emotions:Persuasivo', 'Detalhe de Produto / Serviço:branco', 'Characters Gender:Feminino', 'Visuals Colors:Branco', 'Detalhe de Produto / Serviço:Feminino']
Index(['Audio', 'TotalInsertions', 'TotalLift', 'AvgLift', 'AvgCallToAction',
       'AvgEffectiveMinutes', 'AvgUpliftProportional', 'Nome do Comercial',
       'AdBrandName', 'Secundagem', 'Tags', 'Link do vídeo', 'engneural',
       'foco', 'demanda_cognitiva', 'engneural_5sec', 'foco_5sec',
       'demanda_cognitiva_5sec', 'LOG(TotalLift+1)', 'LOG(AvgLift+1)',
       'LOG(AvgUpliftProportional+1)', 'final', 'lista_tags',
       'Tag_Emotions Emotions:informal', 'Tag_Tone:Informal',
       'Tag_Emotions Sensations:Informal', 'Tag_Characters Gender:Masculino',
       'Tag_Detalhe de Produto / Serviço:masculino',
       'Tag_Emotions Emotio

In [135]:
tentativas = df.copy()
#tentativas = tentativas[tentativas['TotalInsertions'] > 40]
#tentativas = tentativas[tentativas['Tag_Emotions Sensations:Informal'] + tentativas['Tag_Emotions Emotions:Persuasivo'] == 1]
#tentativas = tentativas[(tentativas['Secundagem'] <= 10)]# & (tentativas['Secundagem'] <= 60)]
print(len(tentativas))
print(tentativas.describe())

307
               Audio  TotalInsertions     TotalLift      AvgLift  \
count     307.000000       307.000000    307.000000   307.000000   
mean   820127.742671        31.785016   1040.663974    76.462150   
std     40160.645788        45.389880   2389.992664   229.225811   
min    572123.000000         1.000000      0.170000     0.028333   
25%    822105.500000         5.000000     83.385000     5.417903   
50%    835321.000000        16.000000    265.520000    18.440000   
75%    839885.000000        36.000000    833.525000    57.996111   
max    845475.000000       260.000000  19808.760000  2711.000000   

       AvgCallToAction  AvgEffectiveMinutes  AvgUpliftProportional  \
count       307.000000           307.000000             307.000000   
mean          2.309446             0.739414               2.190101   
std           2.065593             1.248729               7.680930   
min           0.000000             0.000000               0.000000   
25%           1.000000           

In [136]:
# 1. Calcula a matriz de correlação completa entre todas as variáveis
# Por padrão, usa o método Pearson.
matriz_correlacao = tentativas.corr(numeric_only=True)

# 2. Isola a coluna da variável resposta ('clicksperimpression')
# Esta Series agora contém a correlação de cada feature com a variável resposta.
correlacao_com_resposta = matriz_correlacao['LOG(AvgLift+1)']

# 3. (Opcional) Ordena os valores para ver o ranking das features
# Usamos .drop() para remover a autocorrelação (1.0)
correlacao_ordenada = correlacao_com_resposta.drop(['LOG(AvgLift+1)', 'AvgLift', 'TotalLift', 'LOG(TotalLift+1)', 'AvgUpliftProportional',\
        'LOG(AvgUpliftProportional+1)', 'Audio', 'Tag_Tone:Informal', 'Tag_Characters Gender:Masculino']).sort_values(ascending=False)

print("--- Correlação das Features com LOG(AvgLift+1) ---")
print(correlacao_ordenada)

--- Correlação das Features com LOG(AvgLift+1) ---
final                                         1.000000
AvgEffectiveMinutes                           0.365846
Tag_Emotions Sensations:Informal              0.157247
Tag_Emotions Emotions:informal                0.147830
engneural                                     0.114083
Tag_Detalhe de Produto / Serviço:masculino    0.113871
Tag_Emotions Emotions:Persuasivo              0.088497
engneural_5sec                                0.076937
Tag_Detalhe de Produto / Serviço:branco       0.069842
demanda_cognitiva_5sec                        0.064163
Tag_Visuals Colors:Branco                     0.062481
Secundagem                                    0.049030
demanda_cognitiva                             0.028816
Tag_Detalhe de Produto / Serviço:Feminino     0.028253
Tag_Characters Gender:Feminino                0.022921
foco_5sec                                    -0.122306
foco                                         -0.151436
AvgCallToActio

In [137]:
# 1. Calcula a matriz de correlação completa entre todas as variáveis
# Por padrão, usa o método Pearson.
matriz_correlacao = tentativas.corr(numeric_only=True)

# 2. Isola a coluna da variável resposta ('clicksperimpression')
# Esta Series agora contém a correlação de cada feature com a variável resposta.
correlacao_com_resposta = matriz_correlacao['AvgLift']

# 3. (Opcional) Ordena os valores para ver o ranking das features
# Usamos .drop() para remover a autocorrelação (1.0)
correlacao_ordenada = correlacao_com_resposta.drop(['LOG(AvgLift+1)', 'AvgLift', 'TotalLift', 'LOG(TotalLift+1)', 'AvgUpliftProportional',\
        'LOG(AvgUpliftProportional+1)', 'Audio', 'Tag_Tone:Informal', 'Tag_Characters Gender:Masculino']).sort_values(ascending=False)

print("--- Correlação das Features com AvgLift ---")
print(correlacao_ordenada)

--- Correlação das Features com AvgLift ---
final                                         0.581063
AvgEffectiveMinutes                           0.188722
engneural_5sec                                0.034503
Tag_Detalhe de Produto / Serviço:Feminino     0.031832
engneural                                     0.027914
Tag_Characters Gender:Feminino                0.023646
Tag_Detalhe de Produto / Serviço:masculino    0.017862
Tag_Emotions Emotions:informal                0.014026
Secundagem                                    0.003330
Tag_Detalhe de Produto / Serviço:branco       0.000944
Tag_Visuals Colors:Branco                    -0.003656
Tag_Emotions Sensations:Informal             -0.014074
demanda_cognitiva_5sec                       -0.035095
Tag_Emotions Emotions:Persuasivo             -0.043634
demanda_cognitiva                            -0.054670
AvgCallToAction                              -0.055943
foco                                         -0.069442
foco_5sec            

In [147]:
correlacao_parcial = tentativas['engneural'].corr(tentativas['engneural_5sec'])
print(f"Correlação entre engneural: {correlacao_parcial:.4f}")

correlacao_parcial2 = tentativas['foco'].corr(tentativas['foco_5sec'])
print(f"Correlação entre foco: {correlacao_parcial2:.4f}")

correlacao_parcial3 = tentativas['demanda_cognitiva'].corr(tentativas['demanda_cognitiva_5sec'])
print(f"Correlação entre demanda cognitiva: {correlacao_parcial3:.4f}")

Correlação entre engneural: 0.7760
Correlação entre foco: 0.6860
Correlação entre demanda cognitiva: 0.7978


In [138]:
# exemplo de modelo para você testar
import statsmodels.formula.api as smf

df_f = df[df['TotalInsertions'] > 0].copy()
df_f['log_insertions'] = np.log(df_f['TotalInsertions'])

model = smf.ols('final ~ engneural * log_insertions + Q("foco") * log_insertions + Q("demanda_cognitiva") * log_insertions', data=df_f).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                  final   R-squared:                       0.128
Model:                            OLS   Adj. R-squared:                  0.107
Method:                 Least Squares   F-statistic:                     6.265
Date:                Thu, 18 Dec 2025   Prob (F-statistic):           7.22e-07
Time:                        21:33:52   Log-Likelihood:                -543.98
No. Observations:                 307   AIC:                             1104.
Df Residuals:                     299   BIC:                             1134.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

In [139]:
print(df.groupby('AdBrandName')['Audio'].count().sort_values())

AdBrandName
Agro               1
Ambipar            1
Amstel             1
Apple              1
Banco Pan          1
                  ..
Magazine Luiza     9
Havan              9
Amazon            10
Globoplay         11
Ultrafarma        13
Name: Audio, Length: 139, dtype: int64


In [140]:
# import pandas as pd
# import numpy as np
# import statsmodels.formula.api as smf

# # 1) Carregar a base
# grupos = df.copy()

# # Garante que não tem NA nas três variáveis principais
# cols_modelo = ["engneural", "foco", "demanda_cognitiva", "engneural_5sec", "foco_5sec", "demanda_cognitiva_5sec", "final", "AvgLift"]
# grupos = grupos.dropna(subset=cols_modelo)

# def faixa_secundagem(sec):
#     if sec <= 10:
#         return "Curto (≤10s)"
#     elif sec <= 30:
#         return "30s (10–30s)"
#     elif sec <= 60:
#         return "60s (30–60s)"
#     else:
#         return "Longo (≥60s)"

# grupos["Faixa_Secundagem"] = grupos["Secundagem"].apply(faixa_secundagem)

# def faixa_efectiveminutes(x):
#     if x <= 1:
#         return "até 1 min"
#     elif x <= 3:
#         return "entre 1 e 3 min"
#     elif x <= 7:
#         return "entre 3 e 7 min"
#     else:
#         return "entre 7 e 10 min"

# grupos["AvgEffectiveMinutes_faixa"] = grupos["AvgEffectiveMinutes"].apply(faixa_efectiveminutes)

# def faixa_insertions(x):
#     if x <= 40:
#         return "até 40 insercoes"
#     elif x <= 60:
#         return "entre 40 e 60 insercoes"
#     elif x <= 80:
#         return "entre 60 e 80 insercoes"
#     else:
#         return "acima de 80 insercoes"

# grupos["TotalInsertions_faixa"] = grupos["TotalInsertions"].apply(faixa_insertions)


# # 2) Escolher colunas de segmentação (ajuste para o que existe na sua base)
# segment_cols = [
#     "TotalInsertions_faixa",         
#     "Tag_Detalhe de Produto / Serviço:masculino",
#     "AvgEffectiveMinutes_faixa",
#     "Faixa_Secundagem",
#     'Tag_Detalhe de Produto / Serviço:Feminino',
#     'Tag_Characters Gender:Feminino',
#     'Tag_Visuals Colors:Branco',
#     'Tag_Detalhe de Produto / Serviço:branco',
#     'Tag_Emotions Emotions:Persuasivo',
#     'Tag_Emotions Sensations:Informal',
#     'Tag_Emotions Emotions:informal',
#     'AdBrandName'
# ]

# # Se alguma dessas não existir, remova da lista:
# segment_cols = [c for c in segment_cols if c in grupos.columns]

# def regress_por_grupo(grupos, target_col, group_col, min_n=20):
#     """
#     Roda regressão: target_col ~ engneural + foco + demanda_cognitiva
#     para cada valor de group_col.
#     Retorna um DataFrame com R², n, p-valores e correlações com a variável target.
#     """
#     resultados = []
#     for valor, sub in grupos.groupby(group_col):
#         if len(sub) < min_n:
#             continue
        
#         formula = f'{target_col} ~ engneural + Q("foco") + Q("demanda_cognitiva") + engneural_5sec + foco_5sec + demanda_cognitiva_5sec'
#         try:
#             model = smf.ols(formula, data=sub).fit()
#         except Exception as e:
#             print(f"Erro no grupo {group_col}={valor}: {e}")
#             continue

#         # correlações dentro do grupo
#         corr_series = (
#             sub[[target_col, "engneural", "foco", "demanda_cognitiva", "engneural_5sec", "foco_5sec", "demanda_cognitiva_5sec"]]
#             .corr(numeric_only=True)[target_col]
#         )
#         corr_eng = corr_series.get("engneural", np.nan)
#         corr_foco = corr_series.get("foco", np.nan)
#         corr_dem  = corr_series.get("demanda_cognitiva", np.nan)
#         corr_eng_5sec = corr_series.get("engneural_5sec", np.nan)
#         corr_foco_5sec = corr_series.get("foco_5sec", np.nan)
#         corr_dem_5sec  = corr_series.get("demanda_cognitiva_5sec", np.nan)
        
#         resultados.append({
#             "segment_col": group_col,
#             "segment_value": valor,
#             "target": target_col,
#             "n": len(sub),
#             "R2": model.rsquared,
#             "R2_adj": model.rsquared_adj,
#             "p_engneural": model.pvalues.get("engneural", np.nan),
#             "p_Foco": model.pvalues.get('Q("foco")', np.nan),
#             "p_Demanda": model.pvalues.get('Q("demanda_cognitiva")', np.nan),
#             "corr_engneural": corr_eng,
#             "corr_Foco": corr_foco,
#             "corr_Demanda": corr_dem,
#             "p_engneural_5sec": model.pvalues.get("engneural_5sec", np.nan),
#             "p_Foco_5sec": model.pvalues.get("foco_5sec", np.nan),
#             "p_Demanda_5sec": model.pvalues.get("demanda_cognitiva_5sec", np.nan),
#             "corr_engneural_5sec": corr_eng_5sec,
#             "corr_Foco_5sec": corr_foco_5sec,
#             "corr_Demanda_5sec": corr_dem_5sec,
#         })
    
#     if not resultados:
#         return pd.DataFrame(columns=[
#             "segment_col", "segment_value", "target", "n",
#             "R2", "R2_adj",
#             "p_engneural", "p_Foco", "p_Demanda",
#             "corr_engneural", "corr_Foco", "corr_Demanda",
#             "p_engneural_5sec", "p_Foco_5sec", "p_Demanda_5sec",
#             "corr_engneural_5sec", "corr_Foco_5sec", "corr_Demanda_5sec",
#         ])
    
#     df_res = pd.DataFrame(resultados)
#     return df_res.sort_values("R2", ascending=False)

# # 3) Rodar para cada coluna de segmentação, para final e AvgLift
# todos_results = []
# for gcol in segment_cols:
#     res_final = regress_por_grupo(grupos, "final", gcol, min_n=20)
#     res_avg   = regress_por_grupo(grupos, "AvgLift", gcol, min_n=20)
    
#     if not res_final.empty:
#         todos_results.append(res_final)
#     if not res_avg.empty:
#         todos_results.append(res_avg)

# if todos_results:
#     resultados_segmentos = pd.concat(todos_results, ignore_index=True)
# else:
#     resultados_segmentos = pd.DataFrame(columns=[
#         "segment_col", "segment_value", "target", "n",
#         "R2", "R2_adj", "p_engneural", "p_Foco", "p_Demanda"
#     ])

# # 4) Ver os TOP grupos
# # Grupos com R2 >= 0.3 e n >= 20, ordenados por R2
# fortes = resultados_segmentos.query("R2 >= 0.3 and n >= 20").sort_values("R2", ascending=False)

# print("Grupos com R2 >= 0.3:")
# print(fortes.head(30))

# # Se quiser também ver os melhores por target, mesmo que R2 seja menor:
# print("\nTop 20 grupos para 'final':")
# print(resultados_segmentos[resultados_segmentos["target"]=="final"].head(20))

# print("\nTop 20 grupos para 'AvgLift':")
# print(resultados_segmentos[resultados_segmentos["target"]=="AvgLift"].head(20))


# fortes.to_csv('fortes.csv', sep = ';', decimal = ',')
# resultados_segmentos.to_csv('resultados_segmentos.csv', sep = ';', decimal = ',')


In [144]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# 1) Carregar a base
grupos = df.copy()

# Garante que não tem NA nas três variáveis principais
cols_modelo = ["engneural", "foco", "demanda_cognitiva","engneural_5sec", "foco_5sec", "demanda_cognitiva_5sec", "final", "AvgLift"]
grupos = grupos.dropna(subset=cols_modelo)

def faixa_secundagem(sec):
    if sec <= 10:
        return "Curto (≤10s)"
    elif sec <= 30:
        return "30s (10–30s)"
    elif sec <= 60:
        return "60s (30–60s)"
    else:
        return "Longo (≥60s)"

grupos["Faixa_Secundagem"] = grupos["Secundagem"].apply(faixa_secundagem)

def faixa_efectiveminutes(x):
    if x <= 1:
        return "até 1 min"
    elif x <= 3:
        return "entre 1 e 3 min"
    elif x <= 7:
        return "entre 3 e 7 min"
    else:
        return "entre 7 e 10 min"

grupos["AvgEffectiveMinutes_faixa"] = grupos["AvgEffectiveMinutes"].apply(faixa_efectiveminutes)

def faixa_insertions(x):
    if x <= 40:
        return "até 40 insercoes"
    elif x <= 60:
        return "entre 40 e 60 insercoes"
    elif x <= 80:
        return "entre 60 e 80 insercoes"
    else:
        return "acima de 80 insercoes"

grupos["TotalInsertions_faixa"] = grupos["TotalInsertions"].apply(faixa_insertions)


# 2) Escolher colunas de segmentação (ajuste para o que existe na sua base)
segment_cols = [
    "TotalInsertions_faixa",         
    "Tag_Detalhe de Produto / Serviço:masculino",
    "AvgEffectiveMinutes_faixa",
    "Faixa_Secundagem",
    'Tag_Detalhe de Produto / Serviço:Feminino',
    'Tag_Characters Gender:Feminino',
    'Tag_Visuals Colors:Branco',
    'Tag_Detalhe de Produto / Serviço:branco',
    'Tag_Emotions Emotions:Persuasivo',
    'Tag_Emotions Sensations:Informal',
    'Tag_Emotions Emotions:informal',
    'AdBrandName'
]

# Se alguma dessas não existir, remova da lista:
segment_cols = [c for c in segment_cols if c in grupos.columns]

def regress_por_grupo(grupos, target_col, group_col, min_n=20):
    """
    Roda regressão: target_col ~ engneural + foco + demanda_cognitiva
    para cada valor de group_col.
    Retorna um DataFrame com R², n, p-valores e correlações com a variável target.
    """
    resultados = []
    for valor, sub in grupos.groupby(group_col):
        if len(sub) < min_n:
            continue
        
        formula = f'{target_col} ~ engneural + Q("foco") + Q("demanda_cognitiva") + engneural_5sec + Q("foco_5sec") + Q("demanda_cognitiva_5sec")'
        try:
            model = smf.ols(formula, data=sub).fit()
        except Exception as e:
            print(f"Erro no grupo {group_col}={valor}: {e}")
            continue

        # correlações dentro do grupo
        corr_series = (
            sub[[target_col, "engneural", "foco", "demanda_cognitiva", "engneural_5sec", "foco_5sec", "demanda_cognitiva_5sec"]]
            .corr(numeric_only=True)[target_col]
        )
        corr_eng = corr_series.get("engneural", np.nan)
        corr_foco = corr_series.get("foco", np.nan)
        corr_dem  = corr_series.get("demanda_cognitiva", np.nan)
        corr_eng_5sec = corr_series.get("engneural_5sec", np.nan)
        corr_foco_5sec = corr_series.get("foco_5sec", np.nan)
        corr_dem_5sec  = corr_series.get("demanda_cognitiva_5sec", np.nan)
        

        resultados.append({
            "segment_col": group_col,
            "segment_value": valor,
            "target": target_col,
            "n": len(sub),
            "R2": model.rsquared,
            "R2_adj": model.rsquared_adj,
            "p_engneural": model.pvalues.get("engneural", np.nan),
            "p_Foco": model.pvalues.get('Q("foco")', np.nan),
            "p_Demanda": model.pvalues.get('Q("demanda_cognitiva")', np.nan),
            "corr_engneural": corr_eng,
            "corr_Foco": corr_foco,
            "corr_Demanda": corr_dem,
            "p_engneural_5sec": model.pvalues.get("engneural_5sec", np.nan),
            "p_Foco_5sec": model.pvalues.get('Q("foco_5sec")', np.nan),
            "p_Demanda_5sec": model.pvalues.get('Q("demanda_cognitiva_5sec")', np.nan),
            "corr_engneural_5sec": corr_eng_5sec,
            "corr_Foco_5sec": corr_foco_5sec,
            "corr_Demanda_5sec": corr_dem_5sec,
        })
    
    if not resultados:
        return pd.DataFrame(columns=[
            "segment_col", "segment_value", "target", "n",
            "R2", "R2_adj",
            "p_engneural", "p_Foco", "p_Demanda",
            "corr_engneural", "corr_Foco", "corr_Demanda",
            "p_engneural_5sec", "p_Foco_5sec", "p_Demanda_5sec",
            "corr_engneural_5sec", "corr_Foco_5sec", "corr_Demanda_5sec",
        ])
    
    df_res = pd.DataFrame(resultados)
    return df_res.sort_values("R2", ascending=False)

# 3) Rodar para cada coluna de segmentação, para final e AvgLift
todos_results = []
for gcol in segment_cols:
    res_final = regress_por_grupo(grupos, "final", gcol, min_n=20)
    res_avg   = regress_por_grupo(grupos, "AvgLift", gcol, min_n=20)
    
    if not res_final.empty:
        todos_results.append(res_final)
    if not res_avg.empty:
        todos_results.append(res_avg)

if todos_results:
    resultados_segmentos = pd.concat(todos_results, ignore_index=True)
else:
    resultados_segmentos = pd.DataFrame(columns=[
        "segment_col", "segment_value", "target", "n",
        "R2", "R2_adj", "p_engneural_5sec", "p_Foco_5sec", "p_Demanda_5sec"
    ])

# 4) Ver os TOP grupos
# Grupos com R2 >= 0.3 e n >= 20, ordenados por R2
fortes = resultados_segmentos.query("R2 >= 0.3 and n >= 20").sort_values("R2", ascending=False)

print("Grupos com R2 >= 0.3:")
print(fortes.head(30))

# Se quiser também ver os melhores por target, mesmo que R2 seja menor:
print("\nTop 20 grupos para 'final':")
print(resultados_segmentos[resultados_segmentos["target"]=="final"].head(20))

print("\nTop 20 grupos para 'AvgLift':")
print(resultados_segmentos[resultados_segmentos["target"]=="AvgLift"].head(20))


fortes.to_csv('fortes.csv', sep = ';', decimal = ',')
resultados_segmentos.to_csv('resultados_segmentos.csv', sep = ';', decimal = ',')


Grupos com R2 >= 0.3:
              segment_col            segment_value   target   n        R2  \
0   TotalInsertions_faixa  entre 40 e 60 insercoes    final  20  0.475314   
14       Faixa_Secundagem             60s (30–60s)    final  26  0.467553   
3   TotalInsertions_faixa  entre 40 e 60 insercoes  AvgLift  20  0.417837   

      R2_adj  p_engneural    p_Foco  p_Demanda  corr_engneural  corr_Foco  \
0   0.233152     0.355818  0.916184   0.151211       -0.188385  -0.017789   
14  0.299412     0.458774  0.360956   0.032672       -0.259587   0.090419   
3   0.149146     0.414394  0.701696   0.939519       -0.402553   0.064152   

    corr_Demanda  p_engneural_5sec  p_Foco_5sec  p_Demanda_5sec  \
0      -0.418134          0.402242     0.462050        0.369677   
14     -0.628889          0.684093     0.479035        0.765903   
3      -0.456541          0.602118     0.165003        0.593256   

    corr_engneural_5sec  corr_Foco_5sec  corr_Demanda_5sec  
0             -0.260027       

In [145]:
from itertools import combinations
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

def regress_por_grupo_duplo(grupos, target_col, group_col1, group_col2, min_n=20):
    """
    Roda regressão: target_col ~ engneural + foco + demanda_cognitiva
                    + engneural_5sec + foco_5sec + demanda_cognitiva_5sec
    para cada combinação de (group_col1, group_col2).
    Só guarda grupos com n >= min_n.
    """
    resultados = []
    
    for (val1, val2), sub in grupos.groupby([group_col1, group_col2]):
        if len(sub) < min_n:
            continue
        
        # Fórmula com as 6 variáveis explicativas
        formula = (
            f'{target_col} ~ '
            f'engneural + foco + demanda_cognitiva + '
            f'engneural_5sec + foco_5sec + demanda_cognitiva_5sec'
        )
        
        try:
            model = smf.ols(formula, data=sub).fit()
        except Exception as e:
            print(f"Erro no grupo {group_col1}={val1}, {group_col2}={val2}: {e}")
            continue

        # Correlações dentro do grupo com a variável target
        cols_corr = [
            target_col,
            "engneural", "foco", "demanda_cognitiva",
            "engneural_5sec", "foco_5sec", "demanda_cognitiva_5sec"
        ]
        # Garante que todas existem no sub antes de selecionar
        cols_corr = [c for c in cols_corr if c in sub.columns]
        
        corr_series = (
            sub[cols_corr]
            .corr(numeric_only=True)[target_col]
        )
        corr_eng       = corr_series.get("engneural", np.nan)
        corr_foco      = corr_series.get("foco", np.nan)
        corr_dem       = corr_series.get("demanda_cognitiva", np.nan)
        corr_eng_5sec  = corr_series.get("engneural_5sec", np.nan)
        corr_foco_5sec = corr_series.get("foco_5sec", np.nan)
        corr_dem_5sec  = corr_series.get("demanda_cognitiva_5sec", np.nan)
        
        resultados.append({
            # agora marcando claramente que é segmentação dupla
            "segment_col": f"{group_col1} & {group_col2}",
            "segment_value": f"{group_col1}={val1} ; {group_col2}={val2}",
            "target": target_col,
            "n": len(sub),
            "R2": model.rsquared,
            "R2_adj": model.rsquared_adj,
            
            # p-valores dos índices "completos"
            "p_engneural": model.pvalues.get("engneural", np.nan),
            "p_Foco": model.pvalues.get("foco", np.nan),
            "p_Demanda": model.pvalues.get("demanda_cognitiva", np.nan),
            
            # correlações dos índices "completos"
            "corr_engneural": corr_eng,
            "corr_Foco": corr_foco,
            "corr_Demanda": corr_dem,
            
            # p-valores dos índices até 5s
            "p_engneural_5sec": model.pvalues.get("engneural_5sec", np.nan),
            "p_Foco_5sec": model.pvalues.get("foco_5sec", np.nan),
            "p_Demanda_5sec": model.pvalues.get("demanda_cognitiva_5sec", np.nan),
            
            # correlações dos índices até 5s
            "corr_engneural_5sec": corr_eng_5sec,
            "corr_Foco_5sec": corr_foco_5sec,
            "corr_Demanda_5sec": corr_dem_5sec,
        })
    
    if not resultados:
        return pd.DataFrame(columns=[
            "segment_col", "segment_value", "target", "n",
            "R2", "R2_adj",
            "p_engneural", "p_Foco", "p_Demanda",
            "corr_engneural", "corr_Foco", "corr_Demanda",
            "p_engneural_5sec", "p_Foco_5sec", "p_Demanda_5sec",
            "corr_engneural_5sec", "corr_Foco_5sec", "corr_Demanda_5sec",
        ])
    
    df_res = pd.DataFrame(resultados)
    return df_res.sort_values("R2", ascending=False)




todos_results = []

# 3A) Segmentação por UMA coluna (como você já tinha)
for gcol in segment_cols:
    res_final = regress_por_grupo(grupos, "final", gcol, min_n=20)
    res_avg   = regress_por_grupo(grupos, "AvgLift", gcol, min_n=20)
    
    if not res_final.empty:
        todos_results.append(res_final)
    if not res_avg.empty:
        todos_results.append(res_avg)

# 3B) Segmentação por DUAS colunas (novidade)
for col1, col2 in combinations(segment_cols, 2):
    res_final_2 = regress_por_grupo_duplo(grupos, "final", col1, col2, min_n=20)
    res_avg_2   = regress_por_grupo_duplo(grupos, "AvgLift", col1, col2, min_n=20)
    
    if not res_final_2.empty:
        todos_results.append(res_final_2)
    if not res_avg_2.empty:
        todos_results.append(res_avg_2)

# Junta tudo
if todos_results:
    resultados_segmentos = pd.concat(todos_results, ignore_index=True)
else:
    resultados_segmentos = pd.DataFrame(columns=[
        "segment_col", "segment_value", "target", "n",
        "R2", "R2_adj", "p_engneural", "p_Foco", "p_Demanda"
    ])


resultados_segmentos.to_csv('resultados_segmentados_2.csv', sep= ';', decimal = ',')

In [146]:
resultados_segmentos_filtrados = resultados_segmentos[#(resultados_segmentos['p_Foco'] < 0.05) | (resultados_segmentos['p_Demanda'] < 0.05) | (resultados_segmentos['p_engneural'] < 0.05) \
                                                      (resultados_segmentos['p_Foco_5sec'] < 0.05) | (resultados_segmentos['p_Demanda_5sec'] < 0.05) | (resultados_segmentos['p_engneural_5sec'] < 0.05) \
                                                        ]

resultados_segmentos_filtrados.to_csv('resultados_segmentados_2_filtrados.csv', sep= ';', decimal = ',')